# CelebA 40k: раздельный экспорт train и val

Создает структуру:

```text
celeba_hourglass_40k/
├── train/
│   ├── images/
│   ├── annotations.csv
│   └── dataset_summary.json
├── val/
│   ├── images/
│   ├── annotations.csv
│   └── dataset_summary.json
├── subset_40k_manifest.csv
├── split_summary.json
├── train.tar.gz
└── val.tar.gz
```

Колонка `source_path` не используется.


In [1]:
from pathlib import Path
import json
import os
import shutil
import tarfile
import numpy as np
import pandas as pd


## 1. Настройки

In [3]:
ROOT_PATH = Path(
    "/Users/baal/Yandex.Disk.localized/WORK/PROg/DLS/"
    "DLS.2026.1.0весна/24-10_FinalProject/CelebA_var"
)

MANIFEST_PATH = ROOT_PATH / "hourglass_subsets_id/subset_40k.csv"
IMAGES_DIR = ROOT_PATH / "celeba_bbox_cropped_all/images"
OUTPUT_ROOT = ROOT_PATH / "hourglass_subsets_id/hourglass_40k_TVsplit/images"

EXPORT_MODE = "copy"  # "copy" или "hardlink"
VAL_FRACTION = 0.10
RANDOM_SEED = 42

CREATE_TRAIN_ARCHIVE = True
CREATE_VAL_ARCHIVE = True


## 2. Чтение и проверка manifest

In [4]:
LANDMARK_NAMES = ["lefteye", "righteye", "nose", "leftmouth", "rightmouth"]

ANNOTATION_COLUMNS = ["image_id", "identity_id"]
for name in LANDMARK_NAMES:
    ANNOTATION_COLUMNS += [f"{name}_x", f"{name}_y"]

REQUIRED_COLUMNS = ANNOTATION_COLUMNS + [
    "crop_width", "crop_height", "all_landmarks_inside"
]

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(MANIFEST_PATH)
if not IMAGES_DIR.is_dir():
    raise FileNotFoundError(IMAGES_DIR)

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"image_id": "string", "identity_id": "Int64"},
)

missing = [c for c in REQUIRED_COLUMNS if c not in manifest.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

manifest["image_id"] = manifest["image_id"].str.strip()
if manifest["image_id"].isna().any() or manifest["image_id"].eq("").any():
    raise ValueError("image_id contains NaN or empty values.")
if manifest["image_id"].duplicated().any():
    raise ValueError("Duplicate image_id values found.")
if manifest["identity_id"].isna().any():
    raise ValueError("identity_id contains NaN.")

numeric_columns = ["crop_width", "crop_height"]
for name in LANDMARK_NAMES:
    numeric_columns += [f"{name}_x", f"{name}_y"]

for column in numeric_columns:
    manifest[column] = pd.to_numeric(manifest[column], errors="coerce")

if manifest[numeric_columns].isna().any().any():
    raise ValueError("Numeric columns contain NaN or invalid values.")

inside = (
    manifest["all_landmarks_inside"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False, "1": True, "0": False})
)
if inside.isna().any():
    raise ValueError("Invalid all_landmarks_inside values.")
if not inside.all():
    raise ValueError(f"{(~inside).sum():,} rows have landmarks outside crop.")

manifest["image_path"] = manifest["image_id"].map(
    lambda image_id: IMAGES_DIR / str(image_id)
)
manifest["image_exists"] = manifest["image_path"].map(Path.is_file)

missing_images = manifest.loc[
    ~manifest["image_exists"], ["image_id", "image_path"]
]

print(f"Rows:              {len(manifest):,}")
print(f"Unique identities: {manifest['identity_id'].nunique():,}")
print(f"Images found:      {manifest['image_exists'].sum():,}")
print(f"Images missing:    {len(missing_images):,}")

if len(missing_images):
    display(missing_images)
    raise FileNotFoundError("Some manifest images are absent.")


Rows:              40,000
Unique identities: 9,402
Images found:      40,000
Images missing:    0


## 3. Identity-disjoint split

In [5]:
identity_sizes = (
    manifest.groupby("identity_id").size().rename("n_images").reset_index()
)

rng = np.random.default_rng(RANDOM_SEED)
identity_sizes = identity_sizes.iloc[
    rng.permutation(len(identity_sizes))
].reset_index(drop=True)

target_val = round(len(manifest) * VAL_FRACTION)
val_identity_ids = []
val_count = 0

for row in identity_sizes.itertuples(index=False):
    if val_count >= target_val:
        break
    new_count = val_count + int(row.n_images)
    if abs(target_val - new_count) <= abs(target_val - val_count) or val_count == 0:
        val_identity_ids.append(int(row.identity_id))
        val_count = new_count

val_identity_ids = set(val_identity_ids)

manifest["split"] = np.where(
    manifest["identity_id"].isin(val_identity_ids),
    "val",
    "train",
)

train_df = manifest.loc[manifest["split"] == "train"].copy().reset_index(drop=True)
val_df = manifest.loc[manifest["split"] == "val"].copy().reset_index(drop=True)

train_ids = set(train_df["identity_id"].astype(int))
val_ids = set(val_df["identity_id"].astype(int))

assert train_ids.isdisjoint(val_ids)
assert len(train_df) + len(val_df) == len(manifest)

print(f"Train images:       {len(train_df):,}")
print(f"Validation images:  {len(val_df):,}")
print(f"Train identities:   {len(train_ids):,}")
print(f"Val identities:     {len(val_ids):,}")
print(f"Identity overlap:   {len(train_ids & val_ids)}")


Train images:       36,000
Validation images:  4,000
Train identities:   8,466
Val identities:     936
Identity overlap:   0


## 4. Экспорт split

In [6]:
def export_split(split_df, split_name):
    split_root = OUTPUT_ROOT / split_name
    images_root = split_root / "images"
    images_root.mkdir(parents=True, exist_ok=True)

    created = 0
    existing = 0

    for position, row in enumerate(split_df.itertuples(index=False), start=1):
        source = IMAGES_DIR / str(row.image_id)
        destination = images_root / str(row.image_id)

        if destination.exists():
            existing += 1
            continue

        if EXPORT_MODE == "hardlink":
            try:
                os.link(source, destination)
            except OSError:
                shutil.copy2(source, destination)
        elif EXPORT_MODE == "copy":
            shutil.copy2(source, destination)
        else:
            raise ValueError("EXPORT_MODE must be 'copy' or 'hardlink'.")

        created += 1

        if position % 5000 == 0 or position == len(split_df):
            print(f"{split_name}: {position:,}/{len(split_df):,}")

    compact_columns = ANNOTATION_COLUMNS + ["crop_width", "crop_height"]
    annotations = split_df[compact_columns].copy()
    annotations.to_csv(split_root / "annotations.csv", index=False)

    counts = split_df["identity_id"].value_counts()
    summary = {
        "split": split_name,
        "images": int(len(split_df)),
        "identities": int(split_df["identity_id"].nunique()),
        "min_images_per_identity": int(counts.min()),
        "max_images_per_identity": int(counts.max()),
        "mean_images_per_identity": float(counts.mean()),
        "images_directory": "images",
        "annotations_file": "annotations.csv",
        "created_files": int(created),
        "already_present": int(existing),
    }

    with open(split_root / "dataset_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    return split_root, images_root, annotations, summary


## 5. Экспорт train и val

In [7]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

train_root, train_images_root, train_annotations, train_summary = export_split(
    train_df, "train"
)
val_root, val_images_root, val_annotations, val_summary = export_split(
    val_df, "val"
)

manifest.drop(
    columns=["image_path", "image_exists"], errors="ignore"
).to_csv(
    OUTPUT_ROOT / "subset_40k_manifest.csv", index=False
)

split_summary = {
    "total_images": int(len(manifest)),
    "total_identities": int(manifest["identity_id"].nunique()),
    "train_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "train_identities": int(len(train_ids)),
    "validation_identities": int(len(val_ids)),
    "identity_overlap": int(len(train_ids & val_ids)),
    "validation_fraction": float(len(val_df) / len(manifest)),
    "random_seed": int(RANDOM_SEED),
}

with open(OUTPUT_ROOT / "split_summary.json", "w", encoding="utf-8") as f:
    json.dump(split_summary, f, ensure_ascii=False, indent=2)

print(OUTPUT_ROOT)


train: 5,000/36,000
train: 10,000/36,000
train: 15,000/36,000
train: 20,000/36,000
train: 25,000/36,000
train: 30,000/36,000
train: 35,000/36,000
train: 36,000/36,000
val: 4,000/4,000
/Users/baal/Yandex.Disk.localized/WORK/PROg/DLS/DLS.2026.1.0весна/24-10_FinalProject/CelebA_var/hourglass_subsets_id/hourglass_40k_TVsplit/images


## 6. Финальная проверка

In [8]:
def verify(split_df, images_root, annotations, name):
    exported = {p.name for p in images_root.iterdir() if p.is_file()}
    expected = set(split_df["image_id"].astype(str))

    missing = sorted(expected - exported)
    unexpected = sorted(exported - expected)

    print(f"{name}: expected={len(expected):,}, exported={len(exported):,}, "
          f"missing={len(missing):,}, unexpected={len(unexpected):,}")

    if missing:
        print("Missing examples:", missing[:20])
    if unexpected:
        print("Unexpected examples:", unexpected[:20])

    assert not missing
    assert not unexpected
    assert len(annotations) == len(split_df)

verify(train_df, train_images_root, train_annotations, "train")
verify(val_df, val_images_root, val_annotations, "val")

assert set(train_df["image_id"]).isdisjoint(set(val_df["image_id"]))
assert train_ids.isdisjoint(val_ids)
assert len(train_df) + len(val_df) == len(manifest)

print("Final consistency checks passed.")


train: expected=36,000, exported=36,000, missing=0, unexpected=0
val: expected=4,000, exported=4,000, missing=0, unexpected=0
Final consistency checks passed.


## 7. Отдельные архивы

In [9]:
def create_archive(split_root, archive_path):
    with tarfile.open(archive_path, "w:gz") as archive:
        archive.add(split_root, arcname=split_root.name)

    size_gb = archive_path.stat().st_size / (1024**3)
    print(f"{archive_path} - {size_gb:.3f} GB")

if CREATE_TRAIN_ARCHIVE:
    create_archive(OUTPUT_ROOT / "train", OUTPUT_ROOT / "train.tar.gz")

if CREATE_VAL_ARCHIVE:
    create_archive(OUTPUT_ROOT / "val", OUTPUT_ROOT / "val.tar.gz")


/Users/baal/Yandex.Disk.localized/WORK/PROg/DLS/DLS.2026.1.0весна/24-10_FinalProject/CelebA_var/hourglass_subsets_id/hourglass_40k_TVsplit/images/train.tar.gz - 1.107 GB
/Users/baal/Yandex.Disk.localized/WORK/PROg/DLS/DLS.2026.1.0весна/24-10_FinalProject/CelebA_var/hourglass_subsets_id/hourglass_40k_TVsplit/images/val.tar.gz - 0.126 GB


## 8. Использование в Colab

```bash
!mkdir -p /content/celeba_hourglass_40k
!tar -xzf /content/drive/MyDrive/train.tar.gz -C /content/celeba_hourglass_40k
!tar -xzf /content/drive/MyDrive/val.tar.gz -C /content/celeba_hourglass_40k
```

```python
from pathlib import Path

DATA_ROOT = Path("/content/celeba_hourglass_40k")

TRAIN_ROOT = DATA_ROOT / "train"
TRAIN_IMAGES = TRAIN_ROOT / "images"
TRAIN_CSV = TRAIN_ROOT / "annotations.csv"

VAL_ROOT = DATA_ROOT / "val"
VAL_IMAGES = VAL_ROOT / "images"
VAL_CSV = VAL_ROOT / "annotations.csv"
```
